In [1]:
!pip install matplotlib numpy ipywidgets

In [2]:
# Kód pro Pinedův algoritmus

def hrana_funkce(v_start, v_konec, x, y):
    """
    Vypočítá hodnotu hranové funkce E(x, y) pro orientovanou hranu v_start -> v_konec.
    Kladná hodnota (nebo 0) znamená, že bod [x, y] leží vlevo od hrany (nebo na ní).
    """
    xA, yA = v_start
    xB, yB = v_konec
    return (xB - xA) * (y - yA) - (yB - yA) * (x - xA)


def pineda_didactic(v0, v1, v2):
    """
    Pinedův algoritmus pro rasterizaci trojúhelníku.
    Prochází ohraničující obdélník a zaznamenává každý krok.
    """
    # 1. Nalezení ohraničujícího obdélníku (Bounding Box)
    min_x = min(v0[0], v1[0], v2[0])
    max_x = max(v0[0], v1[0], v2[0])
    min_y = min(v0[1], v1[1], v2[1])
    max_y = max(v0[1], v1[1], v2[1])

    steps = []
    vykreslene_pixely = []
    zamitnute_pixely = []
    krok = 0

    # 2. Průchod všemi pixely v ohraničujícím obdélníku
    for y in range(min_y, max_y + 1):
        for x in range(min_x, max_x + 1):
            # Výpočet tří hranových funkcí pro střed pixelu (x, y)
            E0 = hrana_funkce(v0, v1, x, y)
            E1 = hrana_funkce(v1, v2, x, y)
            E2 = hrana_funkce(v2, v0, x, y)

            # Bod je uvnitř, právě když jsou všechny tři funkce nezáporné
            uvnitr = (E0 >= 0) and (E1 >= 0) and (E2 >= 0)

            if uvnitr:
                vykreslene_pixely.append((x, y))
            else:
                zamitnute_pixely.append((x, y))

            steps.append({
                "krok": krok,
                "aktualni_pixel": (x, y),
                "E0": E0,
                "E1": E1,
                "E2": E2,
                "uvnitr": uvnitr,
                "vykreslene_pixely": list(vykreslene_pixely),
                "zamitnute_pixely": list(zamitnute_pixely),
                "bbox": (min_x, max_x, min_y, max_y)
            })
            krok += 1

    return steps

In [3]:
# Kód pro vykreslení algoritmu do grafu

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display

# Pevně dané vrcholy trojúhelníku (proti směru hodinových ručiček)
V0 = (1, 1)
V1 = (5, 2)
V2 = (2, 4)

historie = pineda_didactic(V0, V1, V2)
max_krok = len(historie) - 1

aktualni_krok = 0

# Vytvoření ovládacích prvků
btn_zpet = widgets.Button(description='<- Krok zpět', disabled=True, button_style='info')
btn_vpred = widgets.Button(description='Krok vpřed ->', disabled=False, button_style='info')
lbl_stav = widgets.Label(value=f"Krok: 0 / {max_krok}")
vystup_grafu = widgets.Output(layout=widgets.Layout(min_height='640px'))

def vykresli_aktualni_stav():
    data_kroku = historie[aktualni_krok]
    cur_x, cur_y = data_kroku["aktualni_pixel"]
    E0, E1, E2 = data_kroku["E0"], data_kroku["E1"], data_kroku["E2"]
    uvnitr = data_kroku["uvnitr"]
    min_x, max_x, min_y, max_y = data_kroku["bbox"]

    # Sestavení srozumitelného popisu nad graf
    stav_E = f"Hodnoty hranových funkcí:  E0 = {E0},  E1 = {E1},  E2 = {E2}"
    if uvnitr:
        rozhodnuti = (
            f"Všechny hodnoty jsou nezáporné (≥ 0) → Pixel leží VLEVO od všech tří hran.\n"
            f"Rozhodnutí: Pixel [{cur_x}, {cur_y}] je UVNITŘ trojúhelníku → VYKRESLÍME HO."
        )
    else:
        zaporne = []
        if E0 < 0: zaporne.append(f"E0 ({E0} < 0, vpravo od hrany e0)")
        if E1 < 0: zaporne.append(f"E1 ({E1} < 0, vpravo od hrany e1)")
        if E2 < 0: zaporne.append(f"E2 ({E2} < 0, vpravo od hrany e2)")
        duvod = ", ".join(zaporne)
        rozhodnuti = (
            f"Záporná funkce: {duvod}.\n"
            f"Rozhodnutí: Pixel [{cur_x}, {cur_y}] leží VNĚ trojúhelníku → PŘESKOČÍME HO (×)."
        )

    nadpis = f"Krok {aktualni_krok} z {max_krok} | Testujeme pixel [{cur_x}, {cur_y}] v ohraničujícím obdélníku\n{stav_E}\n{rozhodnuti}"

    with vystup_grafu:
        vystup_grafu.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(9, 6.2))

        # 1. Ohraničující obdélník (Bounding Box)
        bbox_rect = patches.Rectangle(
            (min_x - 0.5, min_y - 0.5),
            (max_x - min_x + 1),
            (max_y - min_y + 1),
            linewidth=2, edgecolor='#6c757d', facecolor='#f8f9fa',
            linestyle=':', zorder=1, label='Ohraničující obdélník (Bounding Box)'
        )
        ax.add_patch(bbox_rect)

        # 2. Vykreslení přijatých (uvnitř) pixelů
        for px, py in data_kroku["vykreslene_pixely"]:
            je_aktualni = (px, py) == (cur_x, cur_y)
            barva = '#ff6b6b' if je_aktualni else '#4ecdc4'
            alfa = 0.9 if je_aktualni else 0.55

            rect = patches.Rectangle((px - 0.5, py - 0.5), 1, 1,
                                     linewidth=1.5, edgecolor='black',
                                     facecolor=barva, alpha=alfa, zorder=2)
            ax.add_patch(rect)

        # 3. Vykreslení zamítnutých (vně) pixelů
        zam_x = [p[0] for p in data_kroku["zamitnute_pixely"]]
        zam_y = [p[1] for p in data_kroku["zamitnute_pixely"]]
        if zam_x:
            ax.scatter(zam_x, zam_y, color='#adb5bd', marker='x', s=50, zorder=3, label='Zamítnutý pixel (vně)')

        # Pokud je aktuální pixel zamítnutý, zvýrazníme jeho čtverec žlutě/oranžově, aby bylo vidět, kde stojíme
        if not uvnitr:
            rect_out = patches.Rectangle((cur_x - 0.5, cur_y - 0.5), 1, 1,
                                         linewidth=2, edgecolor='#e63946',
                                         facecolor='#ffd166', alpha=0.5, zorder=2)
            ax.add_patch(rect_out)

        # Střed aktuálně testovaného pixelu
        ax.scatter([cur_x], [cur_y], color='red', s=45, zorder=5, label=f'Právě testovaný pixel [{cur_x}, {cur_y}]')

        # 4. Ideální matematický trojúhelník a popisky orientovaných hran
        tx = [V0[0], V1[0], V2[0], V0[0]]
        ty = [V0[1], V1[1], V2[1], V0[1]]
        ax.plot(tx, ty, color='#1a535c', linestyle='--', linewidth=2.2, label='Ideální trojúhelník (hrany e0, e1, e2)', zorder=4)
        ax.scatter([V0[0], V1[0], V2[0]], [V0[1], V1[1], V2[1]], color='#1a535c', s=60, zorder=5)

        # Popisky vrcholů a hran přímo v grafu
        ax.annotate(f"V0 {V0}", V0, textcoords="offset points", xytext=(-15, -18), fontweight='bold', color='#1a535c')
        ax.annotate(f"V1 {V1}", V1, textcoords="offset points", xytext=(8, -5), fontweight='bold', color='#1a535c')
        ax.annotate(f"V2 {V2}", V2, textcoords="offset points", xytext=(-15, 10), fontweight='bold', color='#1a535c')

        ax.text(3.1, 1.15, "e0 (V0→V1)", color='#1a535c', fontsize=9, fontweight='bold')
        ax.text(3.7, 3.2, "e1 (V1→V2)", color='#1a535c', fontsize=9, fontweight='bold')
        ax.text(0.55, 2.6, "e2 (V2→V0)", color='#1a535c', fontsize=9, fontweight='bold')

        # Nastavení mřížky (volné místo vpravo nahoře pro legendu)
        grid_max_x = max_x + 3
        grid_max_y = max_y + 3
        ax.set_xticks(range(0, grid_max_x))
        ax.set_yticks(range(0, grid_max_y))
        ax.set_xticks([x - 0.5 for x in range(0, grid_max_x + 1)], minor=True)
        ax.set_yticks([y - 0.5 for y in range(0, grid_max_y + 1)], minor=True)

        ax.grid(which='minor', color='#cccccc', linestyle='-', linewidth=1)
        ax.grid(which='major', color='#eeeeee', linestyle=':', linewidth=0.5)

        ax.set_xlim(-0.5, grid_max_x - 0.5)
        ax.set_ylim(-0.5, grid_max_y - 0.5)
        ax.set_aspect('equal')

        # Panel nad grafem
        ax.set_title(
            nadpis,
            fontsize=10, pad=14, loc='left',
            bbox=dict(boxstyle='round,pad=0.6', facecolor='#f8f9fa', edgecolor='#dee2e6')
        )

        ax.legend(loc='upper right', fontsize=9)
        plt.tight_layout()
        plt.show()

# Obsluha kliknutí na tlačítka
def pri_kliknuti_zpet(b):
    global aktualni_krok
    if aktualni_krok > 0:
        aktualni_krok -= 1
        aktualizuj_ui()

def pri_kliknuti_vpred(b):
    global aktualni_krok
    if aktualni_krok < max_krok:
        aktualni_krok += 1
        aktualizuj_ui()

def aktualizuj_ui():
    btn_zpet.disabled = (aktualni_krok == 0)
    btn_vpred.disabled = (aktualni_krok == max_krok)
    lbl_stav.value = f"Krok: {aktualni_krok} / {max_krok}"
    vykresli_aktualni_stav()

btn_zpet.on_click(pri_kliknuti_zpet)
btn_vpred.on_click(pri_kliknuti_vpred)

# Složení panelu a vykreslení
ovladaci_panel = widgets.HBox([btn_zpet, btn_vpred, lbl_stav])
#display(vystup_grafu, ovladaci_panel)
#vykresli_aktualni_stav()

## Princip a pseudokód Pinedova algoritmu

**Pinedův algoritmus** (využívaný v moderních grafických kartách) převádí problém vyplnění trojúhelníku na testování, zda střed pixelu $[x, y]$ leží uvnitř trojúhelníku. Pokud jsou vrcholy trojúhelníku $V_0, V_1, V_2$ orientovány **proti směru hodinových ručiček**, leží bod uvnitř právě tehdy, když se nachází **vlevo** od všech tří orientovaných hran ($e_0: V_0 \to V_1$, $e_1: V_1 \to V_2$, $e_2: V_2 \to V_0$).

Pro každou hranu $A \to B$ je definována **hranová funkce (Edge Function)**:
$$E(x, y) = (x_B - x_A)(y - y_A) - (y_B - y_A)(x - x_A)$$

* Pokud je $E(x, y) \ge 0$, bod leží vlevo od hrany (nebo přímo na ní).
* Pokud je $E(x, y) < 0$, bod leží vpravo od hrany (vně trojúhelníku).

```text
ALGORITMUS: Pinedův algoritmus pro trojúhelník
VSTUP:  Vrcholy trojúhelníku V0, V1, V2 (proti směru hodinových ručiček)
VÝSTUP: Vykreslené pixely uvnitř trojúhelníku

1. Urči ohraničující obdélník (Bounding Box) trojúhelníku:
   min_x = min(x0, x1, x2),  max_x = max(x0, x1, x2)
   min_y = min(y0, y1, y2),  max_y = max(y0, y1, y2)

2. Pro každý řádek y od min_y do max_y opakuj:
   Pro každý sloupec x od min_x do max_x opakuj:
   
      a) Vypočítej hodnoty hranových funkcí pro tři hrany:
         E0 = (x1 - x0)*(y - y0) - (y1 - y0)*(x - x0)
         E1 = (x2 - x1)*(y - y1) - (y2 - y1)*(x - x1)
         E2 = (x0 - x2)*(y - y2) - (y0 - y2)*(x - x2)
         
      b) Otestuj, zda pixel leží uvnitř trojúhelníku:
         POKUD (E0 >= 0) A ZÁROVEŇ (E1 >= 0) A ZÁROVEŇ (E2 >= 0) PAK:
             Vykresli_Pixel(x, y)
         JINAK:
             // Pixel leží vně trojúhelníku -> přeskoč ho
```

In [4]:
display(vystup_grafu, ovladaci_panel)
vykresli_aktualni_stav()

Output(layout=Layout(min_height='640px'))